# SupportMind
## 01 - Exploratory Data Analysis (EDA)

Dataset: IT Helpdesk Tickets

Objectif : explorer la structure, la qualité et la diversité
des tickets informatiques avant l'entraînement des modèles.

In [1]:

from pathlib import Path
import pandas as pd

# Chemin du dataset depuis le dossier notebooks
dataset_path = Path("../data/tickets.csv")

# Chargement du dataset
df = pd.read_csv(dataset_path)

# Afficher les 5 premières lignes
display(df.head())


,ticket_id,ticket_text,category,priority,created_at,resolution_time_hours,resolved_at
0,TKT-0001,otp for two-factor auth not arriving on my reg...,Access/Login,Low,2024-03-17 16:45:00,13.56,2024-03-18 06:18:36.000000000
1,TKT-0002,not urgent - cannot access coud portal from of...,Network,Low,2024-02-15 20:02:00,11.87,2024-02-16 07:54:12.000000000
2,TKT-0003,several users on my floor - machine boots into...,Software,Medium,2024-03-25 09:42:00,48.60,2024-03-27 10:18:00.000000000
3,TKT-0004,"load balancer marking SRV-01 as down, half the...",Network,Medium,2024-03-10 19:22:00,17.48,2024-03-11 12:50:48.000000000
4,TKT-0005,everyone in the HR department - touch screen o...,Hardware,Medium,2024-01-15 22:20:00,11.00,2024-01-16 09:20:00.000000000


## 1. Dataset Structure

Analyse des dimensions, des types de données et des valeurs manquantes.

In [2]:
# Dimensions du dataset
print("Dimensions :", df.shape)

# Types des colonnes
print("\nTypes des colonnes :")
display(df.dtypes.to_frame(name="Data Type"))

# Valeurs manquantes
print("\nValeurs manquantes :")
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2)
})
display(missing)


Dimensions : (4000, 7)

Types des colonnes :


,Data Type
ticket_id,str
ticket_text,str
category,str
priority,str
created_at,str
resolution_time_hours,float64
resolved_at,str



Valeurs manquantes :


,Missing Values,Missing (%)
ticket_id,0,0.0
ticket_text,0,0.0
category,0,0.0
priority,0,0.0
created_at,0,0.0
resolution_time_hours,0,0.0
resolved_at,0,0.0


## 2. Category Distribution

Analyse du nombre de tickets par catégorie et de leur répartition.

In [12]:
import matplotlib.pyplot as plt

# Nombre de tickets par catégorie
category_counts = df["category"].value_counts()

# Tableau des distributions
category_distribution = pd.DataFrame({
    "Count": category_counts,
    "Percentage (%)": (
        category_counts / len(df) * 100
    ).round(2)
})

display(category_distribution)



,Count,Percentage (%)
category,,
Software,978,24.45
Network,818,20.45
Access/Login,749,18.73
Hardware,628,15.70
Email,495,12.38
Database,332,8.30



## 3. Ticket Text Length Analysis
Analyse de la distribution des longueurs des descriptions
des tickets informatiques.


In [11]:
# Calcul du nombre de mots par ticket
df["word_count"] = df["ticket_text"].str.split().str.len()

# Statistiques descriptives
print("Statistiques sur la longueur des tickets :")
display(df["word_count"].describe().round(2))




Statistiques sur la longueur des tickets :


count    4000.00
mean       13.05
std         3.57
min         6.00
25%        10.00
50%        13.00
75%        15.00
max        28.00
Name: word_count, dtype: float64


## 4. Duplicate Analysis

Analyse des descriptions dupliquées et des éventuelles
contradictions entre les catégories.


In [6]:
# 1. Nombre de descriptions dupliquées
duplicates = df["ticket_text"].duplicated().sum()

print("Descriptions dupliquées :", duplicates)

# 2. Doublons après normalisation simple
normalized_text = (
    df["ticket_text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

normalized_duplicates = normalized_text.duplicated().sum()

print("Doublons après normalisation :", normalized_duplicates)

# 3. Vérifier les contradictions entre catégories
category_counts_per_text = (
    df.assign(normalized_text=normalized_text)
    .groupby("normalized_text")["category"]
    .nunique()
)

conflicting_texts = category_counts_per_text[
    category_counts_per_text > 1
]

print("Descriptions avec catégories contradictoires :",
      len(conflicting_texts))

# 4. Afficher quelques doublons
print("\nExemples de tickets dupliqués :")

display(
    df[df["ticket_text"].duplicated(keep=False)]
    [["ticket_text", "category"]]
    .head(10)
)


Descriptions dupliquées : 364
Doublons après normalisation : 364
Descriptions avec catégories contradictoires : 0

Exemples de tickets dupliqués :


,ticket_text,category
0,otp for two-factor auth not arriving on my reg...,Access/Login
3,"load balancer marking SRV-01 as down, half the...",Network
6,"teams stuck on grey loading screen, cannot joi...",Software
15,webcam not detected in teams and zoom on my la...,Hardware
17,"wifi keeps dropping every 3 minutes, very anno...",Network
19,"machine boots into safe mode randomly, very sl...",Software
44,vpn disconnects automatically after 3 minutes ...,Network
47,my desktop machine - email recall feature not ...,Email
64,otp for two-factor auth not arriving on my reg...,Access/Login
76,network printer reception printer not showing ...,Network



## 5. Textual Diversity Analysis

### 5.1 Vocabulary Diversity

Analysis of vocabulary size and lexical diversity
in IT ticket descriptions.


In [7]:
import re
# Vérification des descriptions
assert df["ticket_text"].notna().all(), "Valeurs manquantes détectées"
assert df["ticket_text"].map(
    lambda x: isinstance(x, str)
).all(), "Descriptions non textuelles détectées"

assert df["ticket_text"].str.strip().ne("").all(), \
    "Descriptions vides détectées"

# Récupérer les descriptions
texts = df["ticket_text"]

# Tokenisation adaptée aux termes techniques IT
def tokenize_it(text):
    pattern = r"[a-zA-Z0-9]+(?:[-_][a-zA-Z0-9]+)*"
    return re.findall(pattern, text.lower())

# Tokenisation de toutes les descriptions
tokens = []

for text in texts:
    tokens.extend(tokenize_it(text))

# Calcul des indicateurs
total_tokens = len(tokens)
unique_tokens = len(set(tokens))

lexical_diversity = (
    unique_tokens / total_tokens
    if total_tokens > 0 else 0
)

# Affichage des résultats
print("Nombre total de tokens :", total_tokens)
print("Taille du vocabulaire :", unique_tokens)
print("Diversité lexicale :", round(lexical_diversity, 4))
print("Diversité lexicale (%) :",
      round(lexical_diversity * 100, 2))

# Test de tokenisation technique
example = "SRV-01 wi-fi ERR-500 two-factor VPN_ERROR"

print("\nExemple de tokenisation :")
print(tokenize_it(example))


Nombre total de tokens : 50329
Taille du vocabulaire : 2254
Diversité lexicale : 0.0448
Diversité lexicale (%) : 4.48

Exemple de tokenisation :
['srv-01', 'wi-fi', 'err-500', 'two-factor', 'vpn_error']



### 5.2 Most Frequent Tokens

Analysis of the most frequent tokens in IT ticket descriptions.


In [10]:
from collections import Counter
import matplotlib.pyplot as plt

# Compter les occurrences des tokens
token_counts = Counter(tokens)

# Extraire les 20 tokens les plus fréquents
top_20 = token_counts.most_common(20)

# Afficher les résultats
top_df = pd.DataFrame(
    top_20,
    columns=["Token", "Frequency"]
)

display(top_df)




,Token,Frequency
0,not,1062
1,in,1032
2,my,952
3,on,912
4,to,879
5,this,713
6,after,670
7,please,632
8,and,625
9,access,578


In [14]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter

# Stopwords anglais
stopwords = set(ENGLISH_STOP_WORDS)

# Conserver les négations importantes
stopwords -= {"not", "no", "cannot"}

# Exclure les stopwords uniquement pour l'analyse
informative_tokens = [
    token for token in tokens
    if token not in stopwords
]

# Top 20 des tokens restants
top_informative = Counter(informative_tokens).most_common(20)

informative_df = pd.DataFrame(
    top_informative,
    columns=["Token", "Frequency"]
)

display(informative_df)

,Token,Frequency
0,not,1062
1,access,578
2,need,546
3,no,479
4,laptop,426
5,cannot,411
6,help,361
7,urgent,349
8,update,345
9,users,344


### 5.3 Text Similarity Analysis
Measure similarity between IT ticket descriptions using
TF-IDF and cosine similarity.
Objective: identify near-duplicate descriptions and assess
potential data leakage risks before model training.


In [15]:

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Conserver uniquement les descriptions uniques
unique_texts = df["ticket_text"].drop_duplicates().reset_index(drop=True)

# Représentation TF-IDF
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+(?:[-_]\w+)*\b",
    ngram_range=(1, 2)
)

X_tfidf = vectorizer.fit_transform(unique_texts)

# Recherche des deux voisins les plus proches
# Le premier voisin est normalement le ticket lui-même
nn = NearestNeighbors(
    n_neighbors=2,
    metric="cosine",
    algorithm="brute"
)

nn.fit(X_tfidf)

distances, indices = nn.kneighbors(X_tfidf)

# Similarité avec le voisin distinct le plus proche
nearest_similarity = 1 - distances[:, 1]

# Statistiques
print("Descriptions uniques :", len(unique_texts))
print(
    "Similarité moyenne au voisin le plus proche :",
    round(nearest_similarity.mean(), 4)
)

print(
    "Descriptions avec similarité >= 0.90 :",
    (nearest_similarity >= 0.90).sum()
)

print(
    "Descriptions avec similarité >= 0.80 :",
    (nearest_similarity >= 0.80).sum()
)

# Exemples de descriptions similaires
print("\nExemples de tickets similaires :")

top_indices = np.argsort(nearest_similarity)[-5:][::-1]

for i in top_indices:
    j = indices[i, 1]

    print("\nSimilarité :", round(nearest_similarity[i], 3))
    print("Ticket A :", unique_texts.iloc[i])
    print("Ticket B :", unique_texts.iloc[j])


Descriptions uniques : 3636
Similarité moyenne au voisin le plus proche : 0.7263
Descriptions avec similarité >= 0.90 : 280
Descriptions avec similarité >= 0.80 : 1239

Exemples de tickets similaires :

Similarité : 0.976
Ticket A : not urgent - forgot windows password and machine is now locked
Ticket B : urgent - forgot windows password and machine is now locked

Similarité : 0.976
Ticket A : urgent - forgot windows password and machine is now locked
Ticket B : not urgent - forgot windows password and machine is now locked

Similarité : 0.959
Ticket A : urgent - need to reset mfa, lost access to authenticator app after phone change
Ticket B : need to reset mfa, lost access to authenticator app after phone change

Similarité : 0.959
Ticket A : need to reset mfa, lost access to authenticator app after phone change
Ticket B : urgent - need to reset mfa, lost access to authenticator app after phone change

Similarité : 0.953
Ticket A : laptop randomly shuts off during meetings, very disru